# Notebook 5: The Full RLM Pipeline

This is where everything comes together. In the previous notebooks you built:
- **Notebook 1**: LLM calls (text in → text out)
- **Notebook 2**: Safe code execution sandbox
- **Notebook 3**: Recursive self-calls via `llm_query()`
- **Notebook 4**: Emergent strategies (peeking, grepping, partition+map)

Now we assemble them into a **complete Recursive Language Model** and run it on real tasks.

## The Complete Architecture

```
User Query ──→ RLMEngine
                  │
                  ├─→ Create Sandbox (with context, llm_query, FINAL)
                  ├─→ Call LLM with query + context metadata
                  ├─→ LLM generates Python code
                  ├─→ Sandbox executes code
                  │     ├─→ Code may call llm_query() → recursive sub-call
                  │     ├─→ Code may call FINAL("answer") → done
                  │     └─→ Code may error → retry with feedback
                  └─→ Return RLMResult (answer + recursion tree + stats)
```

## Setup

In [ ]:
import sys
sys.path.insert(0, "..")
import json

from rlm_core import RLMEngine, LLMClient, RecursionNode
from rlm_core.visualizer import tree_to_text, tree_to_graphviz, tree_to_dict

# For demonstration, we use a SimulatedClient
# Replace with LLMClient(base_url="http://localhost:8000/v1", model="Orion-zhen/Qwen3-1.7B-AWQ")
# when you have vLLM running

class SimulatedClient:
    """Simulates intelligent LLM responses for demonstration."""
    def __init__(self):
        self.call_count = 0
        self.total_prompt_tokens = 0
        self.total_completion_tokens = 0
    
    def completion(self, prompt, **kwargs):
        self.call_count += 1
        self.total_prompt_tokens += len(prompt.split()) * 2
        self.total_completion_tokens += 20
        
        class Result:
            def __init__(self, text, pt, ct):
                self.text = text
                self.prompt_tokens = pt
                self.completion_tokens = ct
        
        # Determine response based on query content
        if "secret" in prompt.lower() or "hidden" in prompt.lower() or "code" in prompt.lower():
            code = '''import re
match = re.search(r'secret code is ([A-Z0-9-]+)', context)
if match:
    FINAL(match.group(1))
else:
    FINAL("No secret code found")'''
        elif "how many" in prompt.lower() or "count" in prompt.lower() or "red" in prompt.lower():
            code = '''import json
data = json.loads(context) if isinstance(context, str) and context.strip().startswith('[') else []
if data:
    count = sum(1 for item in data if item.get("color") == "red" and item.get("size") == "large")
    FINAL(f"{count} items are red and large")
else:
    FINAL("Could not parse data")'''
        elif "engineer" in prompt.lower() or "who" in prompt.lower() or "cto" in prompt.lower():
            code = '''import re
# Search for people and their roles
people = re.findall(r'([A-Z][a-z]+ (?:[A-Z]\\.? )?[A-Z][a-z]+).*?(?:lead engineer|CTO|senior)', context)
if people:
    FINAL(people[0])
else:
    FINAL("Person not found in context")'''
        else:
            code = 'FINAL("Query processed successfully")'
        
        return Result(code, len(prompt.split()) * 2, 20)

client = SimulatedClient()
engine = RLMEngine(client=client, max_depth=3)
print("RLM Engine ready!")

## Task 1: Needle-in-a-Haystack

Can the RLM find a hidden fact buried in a long document?

In [ ]:
with open("../data/samples/needle_haystack.txt") as f:
    needle_doc = f.read()

print(f"Document: {len(needle_doc)} chars, ~{len(needle_doc.split())} words")
print(f"First 200 chars: {needle_doc[:200]}...\n")

result = engine.run(
    query="What is the secret code hidden in this document?",
    context=needle_doc
)

print(f"Answer: {result.answer}")
print(f"LLM calls: {result.total_llm_calls}")
print(f"Max depth: {result.max_depth_reached}")
print(f"\n{'='*50}")
print("Recursion Tree:")
print(tree_to_text(result.root_node))

## Task 2: Aggregation

Can the RLM count items matching a complex filter across a dataset?

In [ ]:
with open("../data/samples/aggregation_items.json") as f:
    items = json.load(f)

agg_context = json.dumps(items)
print(f"Dataset: {len(items)} items, {len(agg_context)} chars\n")

result_agg = engine.run(
    query="How many items in the dataset are both red AND large?",
    context=agg_context
)

print(f"Answer: {result_agg.answer}")
print(f"LLM calls: {result_agg.total_llm_calls}")

# Ground truth verification
actual = len([i for i in items if i["color"] == "red" and i["size"] == "large"])
print(f"\nGround truth: {actual} items are red AND large")
print(f"\n{'='*50}")
print("Recursion Tree:")
print(tree_to_text(result_agg.root_node))

## Task 3: Multi-hop QA

Can the RLM connect facts across multiple documents?

In [ ]:
import os

docs = []
doc_dir = "../data/samples/multihop_docs"
for fname in sorted(os.listdir(doc_dir)):
    if fname.endswith('.txt'):
        with open(os.path.join(doc_dir, fname)) as f:
            docs.append(f.read())

multi_context = "\n\n---DOCUMENT BOUNDARY---\n\n".join(docs)
print(f"Loaded {len(docs)} documents, {len(multi_context)} total chars\n")

result_multi = engine.run(
    query="Who is the lead engineer at the company that made the NovaPad?",
    context=multi_context
)

print(f"Answer: {result_multi.answer}")
print(f"LLM calls: {result_multi.total_llm_calls}")
print(f"\nExpected: Dr. James Park")
print(f"\n{'='*50}")
print("Recursion Tree:")
print(tree_to_text(result_multi.root_node))

## Visualization: Graphviz Recursion Trees

For complex executions, we can render the recursion tree as a graph diagram:

In [ ]:
# Generate Graphviz DOT for the needle-in-haystack execution
dot = tree_to_graphviz(result.root_node)
print("=== Graphviz DOT (paste into https://dreampuf.github.io/GraphvizOnline/) ===")
print(dot)

# Try to render inline if graphviz is installed
try:
    import graphviz
    graph = graphviz.Source(dot)
    display(graph)
except ImportError:
    print("\n(Install graphviz Python package for inline rendering: pip install graphviz)")
except Exception as e:
    print(f"\n(Graphviz rendering not available: {e})")
    print("Copy the DOT code above and paste into an online Graphviz viewer")

## Cost Analysis

How many tokens does each task use? Let's compare:

In [ ]:
import matplotlib.pyplot as plt

# Collect stats
tasks = ["Needle-in-Haystack", "Aggregation", "Multi-hop QA"]
results = [result, result_agg, result_multi]
llm_calls = [r.total_llm_calls for r in results]
max_depths = [r.max_depth_reached for r in results]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# LLM calls per task
axes[0].bar(tasks, llm_calls, color=['#4CAF50', '#2196F3', '#FF9800'])
axes[0].set_title("LLM Calls per Task")
axes[0].set_ylabel("Number of Calls")

# Max depth per task  
axes[1].bar(tasks, max_depths, color=['#4CAF50', '#2196F3', '#FF9800'])
axes[1].set_title("Max Recursion Depth per Task")
axes[1].set_ylabel("Depth")

plt.tight_layout()
plt.savefig("../data/cost_analysis.png", dpi=100, bbox_inches='tight')
plt.show()
print("Chart saved to data/cost_analysis.png")

## Vanilla vs RLM Comparison

What happens if we just stuff the entire context into the prompt (no recursion)?

In [ ]:
# Vanilla approach: stuff everything in the prompt
def vanilla_query(client, question, context, max_context=2000):
    """Traditional approach: stuff context into prompt."""
    truncated = context[:max_context]
    prompt = f"Context: {truncated}\n\nQuestion: {question}\nAnswer concisely:"
    result = client.completion(prompt)
    return result.text

# Compare on needle-in-haystack
vanilla_client = SimulatedClient()
rlm_client = SimulatedClient()

print("=== Vanilla (stuff context in prompt) ===")
vanilla_answer = vanilla_query(vanilla_client, "What is the secret code?", needle_doc)
print(f"Answer: {vanilla_answer}")
print(f"Tokens used: {vanilla_client.total_prompt_tokens + vanilla_client.total_completion_tokens}")

print(f"\n=== RLM (recursive decomposition) ===")
rlm_engine = RLMEngine(client=rlm_client, max_depth=3)
rlm_result = rlm_engine.run("What is the secret code?", needle_doc)
print(f"Answer: {rlm_result.answer}")
print(f"LLM calls: {rlm_result.total_llm_calls}")
print(f"\nKey insight: RLM processes context programmatically rather than stuffing it all into the prompt.")
print(f"For very long contexts (100K+ tokens), vanilla would exceed the context window entirely.")

## Exercise

Create your own task! Write a custom document and question, then run it through the RLM:

In [ ]:
# YOUR EXERCISE: Create a custom task
custom_context = """
Put your own text here. It could be:
- A long article with a hidden fact
- A dataset to analyze
- Multiple documents to cross-reference
"""

custom_query = "Your question about the context"

# Run it!
result = engine.run(query=custom_query, context=custom_context)
print(f"Answer: {result.answer}")
print(f"\nRecursion Tree:")
print(tree_to_text(result.root_node))

## Summary: The Core Track

Congratulations! You've built a complete Recursive Language Model from scratch:

| Component | What it does | Built in |
|-----------|-------------|----------|
| **LLMClient** | Sends prompts, gets completions | Notebook 1 |
| **Sandbox** | Safely executes LLM-generated code | Notebook 2 |
| **llm_query()** | Enables recursive self-calls | Notebook 3 |
| **Strategies** | Emergent patterns the model discovers | Notebook 4 |
| **RLMEngine** | Orchestrates everything | Notebook 5 |

## What's Next?

The **Foundations Track** (Notebooks 6-8) explores the conceptual landscape:
- Notebook 6: How RLMs relate to Chain-of-Thought and Tree-of-Thought
- Notebook 7: Why long context is expensive (quantization, KV cache)
- Notebook 8: How RLMs relate to function calling and tool use

The **Comparisons Track** (Notebooks 9-11) compares RLMs to alternatives:
- Notebook 9: RLM vs RAG
- Notebook 10: RLM vs ReAct agents
- Notebook 11: RLM and DSPy connection